In [9]:
from fungiclef.transforms import get_transforms
import pandas as pd

import torch

import timm

from PIL import Image
import os

import cv2

from tqdm import tqdm
from fungiclef.dataset import ImageMetadataDataset
import numpy as np
from torch.utils.data import DataLoader


In [10]:
train_df = pd.read_parquet("../train.pq")
val_df = pd.read_parquet("../val.pq")

In [11]:
train_df = pd.read_parquet("../train.pq")[:1000]
val_df = pd.read_parquet("../val.pq")[:1000]
val_df = train_df
_df = pd.concat((train_df, val_df))

DIM = 518
BASE_PATH = "../data/DF_FULL"

transforms = get_transforms(data="valid", width=DIM, height=DIM)

valid_dataset = ImageMetadataDataset(
    _df, local_filepath="../data/DF_FULL/", transform=transforms)

loader = DataLoader(valid_dataset, batch_size=3, shuffle=False, num_workers=8)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = timm.create_model("timm/vit_large_patch14_reg4_dinov2.lvd142m", pretrained=True)
model = model.to(device)
model.eval()

all_embs = []
for data in tqdm(loader):
    
    img, metadata, label = data
    img = img.to(device)

    emb = model.forward(img)

    all_embs.append(emb.detach().cpu().numpy())

all_embs = np.vstack(all_embs)

emb_df = pd.DataFrame(_df.image_path)

embs_list = [x for x in all_embs]
emb_df['embedding'] = embs_list
emb_df.to_parquet('dinov2_1024r.pq', index=False)

  2%|▏         | 14/667 [00:02<01:50,  5.90it/s]


KeyboardInterrupt: 

In [3]:
train_df = pd.read_parquet("../train.pq")[:1000]
val_df = pd.read_parquet("../val.pq")[:1000]
val_df = train_df
_df = pd.concat((train_df, val_df))

DIM = 518
BASE_PATH = "../data/DF_FULL"

transforms = get_transforms(data="valid", width=DIM, height=DIM)

valid_dataset = ImageMetadataDataset(
    _df, local_filepath="../data/DF_FULL/", transform=transforms)

loader = DataLoader(valid_dataset, batch_size=3, shuffle=False, num_workers=8)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = timm.create_model("timm/vit_large_patch14_reg4_dinov2.lvd142m", pretrained=True)
model = model.to(device)
model.eval()


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): 

In [11]:
emb_df = pd.DataFrame(_df.image_path)


In [9]:
pd.read_parquet('dinov2_1024r.pq')

,image_path,embedding
0,2238546328-30620.JPG,"[0.005949932, 0.1469564, -0.49620047, -0.11075..."
1,2558871973-53941.JPG,"[-0.65260863, -0.0013587554, 0.5044832, -1.605..."
2,2238503501-245559.JPG,"[0.5572929, 0.28234318, 0.15551983, 0.96624637..."
3,2446759075-197643.JPG,"[-0.017160544, -1.2300191, -0.3357499, 1.29131..."
4,2238472345-167057.JPG,"[0.3639089, 0.3726992, 0.7560097, 1.3864249, 0..."
...,...,...
995,2238538773-29811.JPG,"[0.14986609, -0.13120933, -0.06765176, 0.34687..."
996,2533934432-350540.JPG,"[-0.2388607, 0.09685287, 0.24587709, -0.679718..."
997,2237936984-78046.JPG,"[-0.76126623, -0.040133193, -0.19653559, -0.61..."
998,2446761440-198270.JPG,"[-0.22786197, -0.37157306, -0.5937109, -0.5741..."


In [13]:
all_embs[0]

array([[ 5.9499322e-03,  1.4695640e-01, -4.9620047e-01, ...,
         8.3745509e-01, -2.2710411e-01, -9.1862476e-01],
       [-6.5260863e-01, -1.3587554e-03,  5.0448322e-01, ...,
         9.7527003e-01, -2.7819359e+00,  1.9330758e-01],
       [ 5.5729288e-01,  2.8234318e-01,  1.5551983e-01, ...,
        -2.8082486e-02,  6.2083673e-01, -3.7461355e-02]], dtype=float32)

In [32]:
src = torch.Tensor([all_embs[0]])

In [23]:
t = nn.TransformerEncoderLayer(d_model=1024, nhead=8)

In [24]:
encoder = nn.TransformerEncoder(t, num_layers=6)

/home/chris/miniconda3/envs/fungiclef/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [35]:
encoder(src)

tensor([[[ 0.8518, -1.3888, -0.0182,  ...,  0.4340, -0.0340,  0.7581],
         [-0.5861, -0.6520,  0.1465,  ...,  0.2417,  0.3388,  1.0260],
         [-0.6824,  0.6260,  0.9012,  ...,  0.5882,  1.7133,  0.5000]]],
       grad_fn=<NativeLayerNormBackward0>)

In [19]:
t.forward(all_embs)

TypeError: Transformer.forward() missing 1 required positional argument: 'tgt'

In [10]:
np.vstack(all_embs).shape

(37776, 1024)

In [4]:
filepaths = list(set(_df['image_path']))

In [7]:
import torch.nn as nn

In [8]:
all_embs

NameError: name 'all_embs' is not defined

 11%|█         | 9444/88717 [27:29<3:50:47,  5.72it/s]


KeyboardInterrupt: 

In [15]:
e = emb.detach().cpu().numpy()

In [19]:
np.vstack([e, e]).shape

(32, 1024)

In [4]:
embeddings = []

i = 0
c = 0
save_every = 25

# # for data in tqdm(valid_dataset):
for i in tqdm(range(-866, 0)):

    img, metadata, label = valid_dataset[i]
    img = img.to(device)
    emb = model.forward(img.unsqueeze(0))
    embeddings.append(emb.detach().cpu().numpy())

    # c += 1

    # if c % save_every == 0:
    #     with open (f'../dinov2_1024/emb_{i}.npz', 'wb') as f:
    #         np.savez(f, *embeddings)
    #     c = 0
    #     i += 1
    #     embeddings = []


100%|██████████| 866/866 [01:59<00:00,  7.22it/s]


In [5]:
with open (f'../dinov2_1024/emb_354.npz', 'wb') as f:
        np.savez(f, *embeddings)
    #     c = 0
    #     i += 1
    #     embeddings = []

In [10]:
import numpy 
with open (f'../dinov2_1024/emb_0.npz', 'rb') as f:
    g = np.load(f)
    k = [j for j in g.values()]
    j = np.array(k).squeeze()

In [1]:
import os
import numpy as np
embs = []

for npz in range(355):
    with open(f'../dinov2_1024/emb_{npz}.npz', 'rb') as f:
        g = np.load(f)
        k = [j for j in g.values()]
        j = np.array(k).squeeze()
        embs.append(j)


In [17]:
emb_df = pd.DataFrame(_df.image_path)

In [3]:
full_embs = np.vstack(embs)

In [4]:
embs_list = [x for x in full_embs]

In [15]:
import pandas as pd 
pd.Series(embs_list)[0]
embs_list[-1]

array([ 0.37485263, -1.1768826 , -0.03787116, ..., -0.13209707,
        0.5447676 ,  1.3403261 ], dtype=float32)

In [14]:
img, metadata, label = valid_dataset[-1]
# img = img.to(device)
model.forward(img.unsqueeze(0))

tensor([[ 0.3748, -1.1769, -0.0379,  ..., -0.1321,  0.5448,  1.3403]],
       grad_fn=<SelectBackward0>)

In [19]:

embs_list = [x for x in full_embs]
emb_df['embedding'] = embs_list
emb_df.to_parquet('dinov2_1024.pq')

In [ ]:
np.array(k).shapea

(100, 1, 768)

In [ ]:
np.array(k).shapea


(100, 1, 768)

In [ ]:
start = time.time()

for img_path in tqdm(train_df.image_path[:100]):

    images = [Image.open(os.path.join(BASE, img_path.replace("jpg", "JPG")))]
    model_inputs = processor(images=images, return_tensors="pt")

    m = model_inputs['pixel_values']
    o = model(m)
    o.pooler_output.shape

  0%|          | 0/100 [00:00<?, ?it/s]


AssertionError: Input height (224) doesn't match model (518).